# Qwen3-4B Sinhala continual pre-training on one NVIDIA B200

Continual pre-training (CPT) of **`Qwen/Qwen3-4B`** on the `isji/sinhala-corpus` dataset
using the Sinhala-extended tokenizer **`isji/Extended-Sinhala-Qwen3`** (151,669 base +
25,187 added = 176,856 tokens). Ported from `llama-scripts/sinllama_4b_cpt_b200.ipynb`
with the Qwen-specific changes below. The final cell pushes the CPT adapter + tokenizer to
`isji/qwen3-4b-cpt`.

## Qwen-specific changes vs. the Llama notebook

1. **Architecture guard accepts `qwen3`** instead of `llama`.
2. **Document separator is `<|endoftext|>`, not `tokenizer.eos_token`.** Qwen's
   `eos_token` is `<|im_end|>` — the *chat turn* terminator. The token Qwen pre-training
   uses as the end-of-document separator is `<|endoftext|>` (both appear in the shipped
   `generation_config` eos list). Packing raw corpus documents with `<|im_end|>` would
   teach a chat-format token in a non-chat role.
3. **Tied-embedding fix (critical).** Qwen3-4B ties input embeddings and `lm_head`
   (one shared matrix). Measured empirically on `peft==0.17.1`: wrapping `embed_tokens`
   with `modules_to_save` silently *unties* them — the trainable copy feeds the input side
   while `lm_head` keeps the frozen original, so the 25,187 new Sinhala rows would stay at
   mean-init in the output head and the model could never learn to *generate* new tokens.
   The model cell re-ties `lm_head` to the trainable copy after PEFT wrapping and asserts
   the pointers match. (Verified: gradients then flow through both paths into the shared
   matrix, and the merge → save → reload cycle reproduces a correctly tied model.)
4. **No BOS token.** Qwen has `bos_token_id=None`; nothing is prepended during packing.
   The base-vs-extended special-token comparison still passes (None == None).
5. **New-embedding scale:** +25,187 rows × 2,560 hidden on ONE tied matrix ≈ 64.5M new
   parameters, initialized by mean-resizing and trained via `modules_to_save`, alongside
   LoRA on all attention/MLP projections.
6. **eos metadata alignment.** `isji/Extended-Sinhala-Qwen3` was built from
   `Qwen3-4B-Base`, whose config marks `<|endoftext|>` as eos; the instruct target uses
   `<|im_end|>`. Vocabulary and IDs are identical — the validation cell adopts the target
   model's eos so the pushed tokenizer matches the model's chat convention.

## Base-model note (deliberate choice)

`Qwen/Qwen3-4B` is the instruction-tuned hybrid checkpoint. CPT on raw text erodes chat
behaviour (catastrophic forgetting of instruction tuning) — in this project's pipeline
that is accepted because the QA fine-tuning notebook
(`qwen3-4b-sinhala-qa-finetuning-poc.ipynb`) is the instruction-recovery step run on top
of this checkpoint. For a SinLlama-faithful arm on a pristine base LM instead, change one
line: `MODEL_ID = "Qwen/Qwen3-4B-Base"`.

## B200 choices in this notebook

- The notebook **does not install or replace PyTorch**. The image must already contain a
  Blackwell-capable build (PyTorch ≥ 2.7 with CUDA ≥ 12.8; CUDA 13 builds also work).
- Training uses FP32 master weights with BF16 autocast (`bf16=True`) as the safest
  baseline, plus TF32 for remaining FP32 matmuls.
- Attention uses PyTorch SDPA; no external FlashAttention build is compiled.
- The starting context is 2,048 tokens; the LoRA preset keeps 65,536 tokens per optimizer
  update (microbatch 4 × accumulation 8).
- **Gradient checkpointing stays OFF** (as in the Llama notebook) — it costs ~33%
  throughput to save memory this run has to spare.
- **Liger fused kernels are ON**, which is what lets the microbatch stay at 8 (Llama
  parity). The 176,856-token vocabulary would otherwise make the FP32 logits tensor
  21.6 GiB at microbatch 8 and OOM the GPU; Liger's fused linear cross-entropy computes
  the loss in chunks without materializing it, and its fused RMSNorm/SwiGLU/RoPE cut
  activation memory further. If liger-kernel is missing or lacks Qwen3 support, the
  configuration falls back automatically to the verified-safe microbatch 4.
- Expect this run to remain ~1.25× the per-token compute of the Llama-3.2-3B CPT
  (36 layers vs 28, 27% larger vocabulary). That part is inherent to the model and cannot
  be tuned away; the tuning above targets everything else.
- `torch.compile` starts disabled. It is the next lever if more speed is needed (often
  10–30%) — enable `USE_TORCH_COMPILE` and expect several minutes of compilation first.

## 1. Install the Python-layer dependencies

Do not add `torch` here — replacing the provider's CUDA/PyTorch build can remove the
compiled Blackwell kernels. `peft` is pinned to the version the tied-embedding re-tie fix
was verified against.

In [ ]:
%uv pip install -q "transformers==4.57.6" "datasets==4.4.1" "peft==0.17.1" "accelerate==1.11.0" "huggingface_hub==0.36.0" "sentencepiece>=0.2,<0.3" "safetensors>=0.5,<1" "packaging>=24,<26" "liger-kernel>=0.5.4"

## 2. Configuration

Review this cell before spending GPU time. `TOKENIZER_ID` must be a strict vocabulary
extension of the model's own tokenizer — verified token-for-token before training.

In [ ]:
import os
import re
from pathlib import Path

# Set before CUDA initializes (the preflight cell is the first to touch torch). Reduces
# allocator fragmentation, which matters here because the per-step logits tensor is large.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ---- Model and tokenizer ----
MODEL_ID = "Qwen/Qwen3-4B"                     # see intro: "Qwen/Qwen3-4B-Base" for a pristine base LM
TOKENIZER_ID = "isji/Extended-Sinhala-Qwen3"   # Sinhala-extended Qwen3 tokenizer (strict extension)
MODEL_REVISION = "main"                        # pin a commit hash for a fully reproducible run
TRAINING_MODE = "lora"                         # "lora" or "full"
MODEL_LOAD_DTYPE = "float32"                   # FP32 master weights + BF16 autocast; safest baseline
EXPECTED_MODEL_TYPE = "qwen3"

# ---- Data ----
DATASET_REPO = "isji/sinhala-corpus"
DATASET_FILENAME = "sinhala_corpus.txt"
DATA_DIR = Path("./sinhala_data")
SEQUENCE_LENGTH = 2048
VALIDATION_FRACTION = 0.001            # 0.1% held out
MAX_EVAL_SAMPLES = 512                 # keep periodic evaluation inexpensive
# Tokenizing/packing 10.7M documents is CPU-bound and embarrassingly parallel. The Llama
# notebook capped this at 8 workers, which leaves a B200 host's ~129 cores almost entirely
# idle. Scale with the machine. This stage is also cached under CACHE_ROOT, so a rerun that
# keeps that directory skips it completely.
PREPROCESSING_WORKERS = max(1, min(64, (os.cpu_count() or 4) - 4))

# ---- Optimization ----
NUM_TRAIN_EPOCHS = 1.0

# ---- Wall-clock budget ----
# The full corpus is ~4,063 optimizer updates (266M tokens). At the measured 0.26 it/s on a
# B200 that is ~4.35 h. Setting TARGET_TRAINING_HOURS caps the run at a fixed wall-clock
# budget by converting it into MAX_STEPS, which the cosine LR schedule then decays over
# properly — unlike interrupting a run, this still ends at a converged learning rate.
# Set to None to train the whole corpus.
#
# For reference, the SinLlama paper continually pre-trained on 304M tokens; a 4 h cap here
# still covers ~245M tokens (92% of the corpus), so the shortened run stays comparable.
TARGET_TRAINING_HOURS = 4.0
OBSERVED_STEPS_PER_SECOND = 0.26       # read off the live progress bar; re-measure per GPU

if TARGET_TRAINING_HOURS is None:
    MAX_STEPS = -1                     # -1 means use NUM_TRAIN_EPOCHS
else:
    MAX_STEPS = int(TARGET_TRAINING_HOURS * 3600 * OBSERVED_STEPS_PER_SECOND)

WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
SEED = 42

# Presets preserve 65,536 tokens/update at sequence length 2,048.
#
# ---- Speed/memory: fused Liger kernels ----
# The 176,856-token vocabulary makes the training-time logits tensor
# microbatch x 2048 x 176,856 x 4 bytes (FP32 cross-entropy), allocated twice (value +
# gradient) = 21.6 GiB at microbatch 8. With ~120 GiB of activations across 36 unchecked
# layers that reaches ~176 GiB and OOMs a 178 GiB B200 — which is why microbatch 8 failed.
#
# Liger's FusedLinearCrossEntropy computes the loss in chunks and never materializes those
# full logits, and its fused RMSNorm/SwiGLU/RoPE cut activation memory further. That is what
# makes microbatch 8 (the Llama notebook's setting) fit, and it adds throughput on top.
# If liger-kernel is unavailable or lacks Qwen3 support, everything falls back to the
# verified-safe microbatch 4 automatically.
USE_LIGER_KERNEL = True


def _liger_supports_qwen3():
    try:
        from liger_kernel.transformers import monkey_patch
    except ImportError:
        return False
    return hasattr(monkey_patch, "apply_liger_kernel_to_qwen3")


LIGER_ACTIVE = USE_LIGER_KERNEL and _liger_supports_qwen3()

# Presets preserve 65,536 tokens/update at sequence length 2,048.
#
# Estimated budget on this GPU (calibrated against a real 47.02 GiB run and the earlier OOM):
#     mb 8, no ckpt, no Liger   ~176 GiB  <- OOM, do not use
#     mb 8, no ckpt, Liger      ~133 GiB  <- fits, fastest        [selected when LIGER_ACTIVE]
#     mb 4, no ckpt, no Liger   ~105 GiB  <- fits, ~15% slower    [fallback]
#     mb 4, ckpt on             ~47 GiB   <- fits, ~33% slower for memory that is not scarce
#
# Gradient checkpointing stays OFF: it trades ~33% throughput for memory this run has to
# spare. Adjust the microbatch only after reading the peak-memory report at the end of the
# training cell, keeping microbatch x accumulation = 32 so tokens/update stays 65,536.
if TRAINING_MODE == "lora":
    PER_DEVICE_TRAIN_BATCH_SIZE = 8 if LIGER_ACTIVE else 4
    GRADIENT_ACCUMULATION_STEPS = 4 if LIGER_ACTIVE else 8
    LEARNING_RATE = 2e-4
    USE_GRADIENT_CHECKPOINTING = False
else:
    # Full CPT trains all 4B parameters: activations and optimizer state are far larger, so
    # checkpointing is the only way this fits.
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 16
    LEARNING_RATE = 2e-5
    USE_GRADIENT_CHECKPOINTING = True

PER_DEVICE_EVAL_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE
AUTO_FIND_BATCH_SIZE = False
USE_TORCH_COMPILE = False              # benchmark only after the uncompiled run is stable

# ---- LoRA only ----
LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ---- Logging, checkpoints, and cost ----
LOGGING_STEPS = 10
EVAL_STEPS = 500
SAVE_STEPS = 500
SAVE_TOTAL_LIMIT = 2
RUN_EVALUATION = True
RESUME_IF_AVAILABLE = True
DATALOADER_WORKERS = max(1, min(8, os.cpu_count() or 4))
HOURLY_GPU_COST_USD = 6.25

model_slug = re.sub(r"[^a-z0-9]+", "-", MODEL_ID.lower()).strip("-")
tokenizer_slug = re.sub(r"[^a-z0-9]+", "-", TOKENIZER_ID.lower()).strip("-")
CACHE_ROOT = Path(f"./cache_{model_slug}_{tokenizer_slug}_seq{SEQUENCE_LENGTH}")
OUTPUT_DIR = Path(f"./output_{model_slug}_sinhala_cpt_{TRAINING_MODE}_seq{SEQUENCE_LENGTH}")
FINAL_ARTIFACT_DIR = OUTPUT_DIR / "final"

# ---- Hugging Face upload ----
PUSH_TO_HUB = True
HF_REPO_ID = "isji/qwen3-4b-cpt"
HF_REPO_PRIVATE = True

assert TRAINING_MODE in {"lora", "full"}
assert MODEL_LOAD_DTYPE in {"float32", "bfloat16"}
assert SEQUENCE_LENGTH % 16 == 0
assert 0 < VALIDATION_FRACTION < 1

tokens_per_update = (
    PER_DEVICE_TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
    * SEQUENCE_LENGTH
)
print(f"Mode                 : {TRAINING_MODE}")
print(f"Model                : {MODEL_ID}@{MODEL_REVISION}")
print(f"Tokenizer            : {TOKENIZER_ID}")
print(f"Sequence length      : {SEQUENCE_LENGTH:,}")
print(f"Microbatch           : {PER_DEVICE_TRAIN_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} accumulation")
print(f"Tokens per update    : {tokens_per_update:,}")
print(f"Gradient checkpoint  : {USE_GRADIENT_CHECKPOINTING}")
if LIGER_ACTIVE:
    print("Liger kernels        : ACTIVE (fused cross-entropy -> microbatch 8 fits)")
elif USE_LIGER_KERNEL:
    print("Liger kernels        : REQUESTED BUT UNAVAILABLE -> falling back to microbatch 4.")
    print("                       Install with: uv pip install 'liger-kernel>=0.5.4'")
else:
    print("Liger kernels        : disabled by config -> microbatch 4")
print(f"Learning rate        : {LEARNING_RATE}")
if MAX_STEPS > 0:
    print(f"Step cap             : {MAX_STEPS:,} updates "
          f"(~{TARGET_TRAINING_HOURS:.1f} h at {OBSERVED_STEPS_PER_SECOND} it/s, "
          f"{MAX_STEPS * tokens_per_update / 1e6:.0f}M tokens)")
else:
    print("Step cap             : none (full corpus, one epoch)")
print(f"Output directory     : {OUTPUT_DIR}")
print(f"Push target          : {HF_REPO_ID} (private={HF_REPO_PRIVATE}, enabled={PUSH_TO_HUB})")

## 3. B200 runtime preflight

In [ ]:
import platform
import subprocess
import torch
import transformers
import datasets
import peft
import accelerate
from packaging.version import Version

assert torch.cuda.is_available(), "CUDA is not available in this notebook runtime."

device_index = torch.cuda.current_device()
device_name = torch.cuda.get_device_name(device_index)
capability = torch.cuda.get_device_capability(device_index)
total_memory_bytes = torch.cuda.get_device_properties(device_index).total_memory
total_memory_gib = total_memory_bytes / 2**30
torch_version = Version(torch.__version__.split("+")[0])
cuda_version = Version(torch.version.cuda) if torch.version.cuda else Version("0")

print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA runtime : {torch.version.cuda}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Accelerate   : {accelerate.__version__}")
print(f"GPU          : {device_name}")
print(f"Capability   : sm_{capability[0]}{capability[1]}")
print(f"VRAM         : {total_memory_gib:.1f} GiB ({total_memory_bytes / 1e9:.1f} GB)")
print(f"Torch arches : {torch.cuda.get_arch_list()}")

if capability >= (10, 0):
    assert torch_version >= Version("2.7"), (
        "Blackwell needs PyTorch >=2.7. Use a provider/NVIDIA image with Blackwell support; "
        "do not repair this by blindly running pip install torch."
    )
    assert cuda_version >= Version("12.8"), (
        "This Blackwell GPU needs a PyTorch build compiled for CUDA >=12.8 (or CUDA 13.x)."
    )

if "B200" not in device_name.upper() and "GB200" not in device_name.upper():
    print("WARNING: this runtime does not identify itself as a B200/GB200.")
if total_memory_gib < 170:
    print("WARNING: less than 170 GiB is visible; reduce the microbatch if this is a partitioned GPU.")

# Run a real BF16 kernel so an incompatible binary fails before model/data downloads.
probe = torch.randn((2048, 2048), device="cuda", dtype=torch.bfloat16)
probe_result = (probe @ probe).float().mean().item()
del probe
torch.cuda.empty_cache()
print(f"BF16 matmul   : OK (mean={probe_result:.6f})")

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free", "--format=csv"],
    check=False,
    capture_output=True,
    text=True,
)
print("\nnvidia-smi:\n" + (smi.stdout.strip() or smi.stderr.strip()))

## 4. Hugging Face authentication

In [ ]:
from huggingface_hub import get_token, notebook_login

if get_token() is None:
    notebook_login()
else:
    print("A Hugging Face token is already available in this environment.")

## 5. Download the Sinhala corpus

In [ ]:
from huggingface_hub import hf_hub_download

DATA_DIR.mkdir(parents=True, exist_ok=True)
data_file = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=DATASET_FILENAME,
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
)
assert data_file.is_file() and data_file.stat().st_size > 0
print(f"Dataset: {data_file.resolve()}")
print(f"Size   : {data_file.stat().st_size / 2**30:.2f} GiB")

## 6. Validate the checkpoint and tokenizer

The extended tokenizer must be a strict extension of the model's own: every base token ID
maps to the same token string, and the special tokens match. This protects the GPU budget
from a mismatched tokenizer upload.

In [ ]:
from transformers import AutoConfig, AutoTokenizer

base_config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if getattr(base_config, "model_type", "") != EXPECTED_MODEL_TYPE:
    raise ValueError(
        f"Expected a {EXPECTED_MODEL_TYPE} checkpoint, got model_type={base_config.model_type!r}. "
        "Review the architecture before using this notebook."
    )

max_positions = getattr(base_config, "max_position_embeddings", None)
if max_positions is not None and SEQUENCE_LENGTH > max_positions:
    raise ValueError(
        f"SEQUENCE_LENGTH={SEQUENCE_LENGTH} exceeds model max_position_embeddings={max_positions}."
    )

base_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True)

base_token_count = len(base_tokenizer)
extended_token_count = len(tokenizer)
if extended_token_count < base_token_count:
    raise ValueError(
        f"Tokenizer has fewer tokens ({extended_token_count:,}) than the base ({base_token_count:,})."
    )

def first_token_id_mismatch(left, right, count, chunk_size=8192):
    for start in range(0, count, chunk_size):
        ids = list(range(start, min(start + chunk_size, count)))
        left_tokens = left.convert_ids_to_tokens(ids)
        right_tokens = right.convert_ids_to_tokens(ids)
        for token_id, left_token, right_token in zip(ids, left_tokens, right_tokens):
            if left_token != right_token:
                return token_id, left_token, right_token
    return None

mismatch = first_token_id_mismatch(base_tokenizer, tokenizer, base_token_count)
if mismatch is not None:
    token_id, base_token, extended_token = mismatch
    raise ValueError(
        "The tokenizer is not a strict extension of this model's tokenizer. "
        f"First mismatch at ID {token_id}: base={base_token!r}, extended={extended_token!r}."
    )

# Qwen has no BOS token; both sides report None and the comparison still holds.
if base_tokenizer.bos_token_id != tokenizer.bos_token_id:
    raise ValueError(
        f"bos_token_id differs: base={base_tokenizer.bos_token_id}, "
        f"extended={tokenizer.bos_token_id}."
    )

# eos metadata: the extension repo was built from Qwen3-4B-Base, whose tokenizer_config sets
# eos to <|endoftext|> (base-LM convention). The CPT target Qwen/Qwen3-4B uses <|im_end|>
# (chat convention). The vocabulary and every token ID are identical — only this metadata
# field differs — so adopt the target model's eos for the tokenizer this run saves and
# pushes, keeping the artifact consistent with the model's own chat template.
if base_tokenizer.eos_token_id != tokenizer.eos_token_id:
    print(
        f"NOTE: aligning eos metadata to the target model: extended tokenizer had "
        f"{tokenizer.eos_token!r} (id {tokenizer.eos_token_id}), adopting "
        f"{base_tokenizer.eos_token!r} (id {base_tokenizer.eos_token_id})."
    )
    tokenizer.eos_token = base_tokenizer.eos_token
    if tokenizer.eos_token_id != base_tokenizer.eos_token_id:
        raise ValueError("Failed to align eos metadata with the base model's tokenizer.")

if tokenizer.eos_token_id is None:
    raise ValueError("The tokenizer must define eos_token_id.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Qwen document separator for pre-training-style packing: <|endoftext|>, NOT the chat
# eos <|im_end|>. Both are in the shipped generation_config eos list; <|endoftext|> is the
# end-of-document token the base model was pre-trained with.
DOCUMENT_SEPARATOR = "<|endoftext|>"
document_separator_id = tokenizer.convert_tokens_to_ids(DOCUMENT_SEPARATOR)
if document_separator_id is None or document_separator_id < 0:
    document_separator_id = tokenizer.eos_token_id
    print(f"WARNING: {DOCUMENT_SEPARATOR} not found; falling back to eos_token_id.")

NEW_TOKEN_COUNT = extended_token_count - base_token_count
print(f"Architecture        : {base_config.architectures}")
print(f"Model shape         : hidden={base_config.hidden_size}, layers={base_config.num_hidden_layers}")
print(f"Tied embeddings     : {getattr(base_config, 'tie_word_embeddings', False)}")
print(f"Maximum positions   : {max_positions:,}")
print(f"Base tokenizer      : {base_token_count:,} tokens")
print(f"Training tokenizer  : {extended_token_count:,} tokens")
print(f"New contiguous IDs  : {NEW_TOKEN_COUNT:,} ({base_token_count:,}..{extended_token_count - 1:,})")
print(f"Document separator  : {DOCUMENT_SEPARATOR} (id {document_separator_id})")
print(f"BOS / EOS / PAD     : {tokenizer.bos_token_id} / {tokenizer.eos_token_id} / {tokenizer.pad_token_id}")
print("Tokenizer ancestry  : strict extension verified over all base IDs")

## 7. Tokenize and pack fixed 2,048-token examples

Documents are tokenized without special tokens, terminated with the Qwen end-of-document
separator, concatenated, and split into fixed `SEQUENCE_LENGTH` chunks. Labels are the
input IDs (standard causal LM); the map results are cached under `CACHE_ROOT`.

In [ ]:
from itertools import chain
from datasets import load_dataset

CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# Every map() below writes its result into CACHE_ROOT and is reused on later runs
# (load_from_cache_file=True), so this cell is slow exactly once per
# model+tokenizer+sequence-length combination. A restart that keeps this directory replays
# from cache in seconds. Note that the cache key includes the tokenizer, so editing the
# tokenizer or its special tokens correctly invalidates it and re-tokenizes.
cache_existed = any(CACHE_ROOT.rglob("*.arrow"))
print(f"Preprocessing cache  : {CACHE_ROOT.resolve()}")
print(f"Cache state          : {'reusing existing arrow files' if cache_existed else 'cold, will tokenize from scratch'}")
print(f"Worker processes     : {PREPROCESSING_WORKERS} (of {os.cpu_count()} cores)")

raw_train = load_dataset(
    "text",
    data_files={"train": str(data_file)},
    cache_dir=str(CACHE_ROOT / "raw"),
)["train"]
if len(raw_train) < 2:
    raise ValueError("The corpus needs at least two text records for a train/validation split.")

raw_splits = raw_train.train_test_split(
    test_size=VALIDATION_FRACTION,
    seed=SEED,
    shuffle=True,
)

def tokenize_documents(batch):
    encoded = tokenizer(
        batch["text"],
        add_special_tokens=False,
        return_attention_mask=False,
        truncation=False,
    )
    encoded["input_ids"] = [
        token_ids + [document_separator_id]
        for token_ids in encoded["input_ids"]
    ]
    return encoded

tokenized = raw_splits.map(
    tokenize_documents,
    batched=True,
    num_proc=PREPROCESSING_WORKERS,
    remove_columns=raw_splits["train"].column_names,
    load_from_cache_file=True,
    desc="Tokenizing documents",
)

def pack_documents(batch):
    all_token_ids = list(chain.from_iterable(batch["input_ids"]))
    usable_length = (len(all_token_ids) // SEQUENCE_LENGTH) * SEQUENCE_LENGTH
    chunks = [
        all_token_ids[start : start + SEQUENCE_LENGTH]
        for start in range(0, usable_length, SEQUENCE_LENGTH)
    ]
    return {"input_ids": chunks, "labels": [chunk.copy() for chunk in chunks]}

packed = tokenized.map(
    pack_documents,
    batched=True,
    batch_size=1000,
    num_proc=PREPROCESSING_WORKERS,
    remove_columns=tokenized["train"].column_names,
    load_from_cache_file=True,
    desc=f"Packing {SEQUENCE_LENGTH}-token examples",
)

train_dataset = packed["train"]
eval_dataset = packed["test"]
if MAX_EVAL_SAMPLES is not None and len(eval_dataset) > MAX_EVAL_SAMPLES:
    eval_dataset = eval_dataset.select(range(MAX_EVAL_SAMPLES))
if len(train_dataset) == 0 or (RUN_EVALUATION and len(eval_dataset) == 0):
    raise ValueError(
        "Packing produced an empty split. Reduce SEQUENCE_LENGTH or increase the validation fraction/corpus size."
    )

train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"Raw train lines      : {len(raw_splits['train']):,}")
print(f"Raw validation lines : {len(raw_splits['test']):,}")
print(f"Packed train chunks  : {len(train_dataset):,}")
print(f"Packed eval chunks   : {len(eval_dataset):,}")
print(f"Train tokens/epoch   : {len(train_dataset) * SEQUENCE_LENGTH:,}")
print("Sample:\n" + tokenizer.decode(train_dataset[0]["input_ids"][:256]))

## 8. Load the model, resize embeddings, configure LoRA — and re-tie the head

Qwen3-4B ties `embed_tokens` and `lm_head` into one matrix. PEFT's `modules_to_save`
breaks that tie (measured on `peft==0.17.1`): the input side trains a copy while the
output head keeps the frozen original — so the new Sinhala token rows would never learn to
be *generated*. After `get_peft_model`, `lm_head.weight` is re-pointed at the trainable
copy and the fix is asserted. Gradients then flow through both the input and output paths
into the single shared matrix, which is exactly the tied-embedding semantics of the base
model.

In [ ]:
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, TaskType, get_peft_model

set_seed(SEED)
load_dtype = torch.float32 if MODEL_LOAD_DTYPE == "float32" else torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=load_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

original_embedding_count = model.get_input_embeddings().num_embeddings
if original_embedding_count != len(tokenizer):
    if original_embedding_count < base_token_count:
        raise ValueError(
            f"Model embeddings ({original_embedding_count:,}) are smaller than its own tokenizer "
            f"({base_token_count:,})."
        )
    model.resize_token_embeddings(len(tokenizer), mean_resizing=True)
    print(f"Resized embeddings: {original_embedding_count:,} -> {len(tokenizer):,}")

input_weight = model.get_input_embeddings().weight
output_weight = model.get_output_embeddings().weight
embeddings_are_tied = input_weight.data_ptr() == output_weight.data_ptr()
print(f"Embeddings tied after resize: {embeddings_are_tied}")

if TRAINING_MODE == "lora":
    available_leaf_names = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
    missing_targets = sorted(set(LORA_TARGET_MODULES) - available_leaf_names)
    if missing_targets:
        raise ValueError(f"LoRA targets missing from this architecture: {missing_targets}")

    modules_to_save = None
    if NEW_TOKEN_COUNT > 0:
        modules_to_save = ["embed_tokens"] if embeddings_are_tied else ["embed_tokens", "lm_head"]

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        modules_to_save=modules_to_save,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    print(f"LoRA modules_to_save : {modules_to_save}")

    # ---- Tied-embedding fix ----
    if embeddings_are_tied and modules_to_save is not None:
        base_model = model.base_model.model
        embed_wrapper = base_model.model.embed_tokens
        trainable_embedding = embed_wrapper.modules_to_save["default"].weight
        head_untied = base_model.lm_head.weight.data_ptr() != trainable_embedding.data_ptr()
        if head_untied:
            base_model.lm_head.weight = embed_wrapper.modules_to_save["default"].weight
            print("Re-tied lm_head to the trainable embedding copy.")
        if base_model.lm_head.weight.data_ptr() != trainable_embedding.data_ptr():
            raise RuntimeError(
                "lm_head is not tied to the trainable embedding copy — new Sinhala tokens "
                "would train on the input side only. Do not start training in this state."
            )
        print("Tie verified: lm_head shares the trainable embedding matrix.")

if USE_GRADIENT_CHECKPOINTING and hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
parameter_bytes = sum(
    parameter.numel() * parameter.element_size() for parameter in model.parameters()
)
estimated_fp32_adam_bytes = trainable_parameters * 8
estimated_fp32_gradient_bytes = trainable_parameters * 4

print(f"Total parameters     : {total_parameters:,}")
print(f"Trainable parameters : {trainable_parameters:,} ({100 * trainable_parameters / total_parameters:.2f}%)")
print(f"Parameter storage    : {parameter_bytes / 2**30:.2f} GiB before Trainer placement")
print(f"Adam moments est.    : {estimated_fp32_adam_bytes / 2**30:.2f} GiB")
print(f"Gradient est.        : {estimated_fp32_gradient_bytes / 2**30:.2f} GiB")
print("Activation memory is additional and depends on batch, sequence length, and attention kernels.")

## 9. Build the Trainer

In [ ]:
import shutil
import time as _time
from transformers import Trainer, TrainerCallback, TrainingArguments, default_data_collator
from transformers.trainer_utils import get_last_checkpoint


class WallClockLimitCallback(TrainerCallback):
    """Hard-stop training once a wall-clock budget is exhausted.

    MAX_STEPS already sizes the run to TARGET_TRAINING_HOURS at the expected throughput, and
    finishing that way is preferable because the cosine schedule decays to completion. This
    callback is the backstop for the case where real throughput is below
    OBSERVED_STEPS_PER_SECOND: it caps the wall clock (and the GPU bill) instead of letting
    the run overshoot. If it fires, the learning rate will not have finished decaying — treat
    that checkpoint as a partial run and lower MAX_STEPS for a clean rerun.
    """

    def __init__(self, limit_hours):
        self.limit_seconds = limit_hours * 3600
        self.started_at = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = _time.perf_counter()
        return control

    def on_step_end(self, args, state, control, **kwargs):
        if self.started_at is None:
            return control
        elapsed = _time.perf_counter() - self.started_at
        if elapsed >= self.limit_seconds:
            print(
                f"\nWall-clock limit of {self.limit_seconds / 3600:.2f} h reached at step "
                f"{state.global_step:,}/{state.max_steps:,} — stopping. The LR schedule did "
                "not complete; lower MAX_STEPS (or raise OBSERVED_STEPS_PER_SECOND) for a "
                "clean run.",
                flush=True,
            )
            control.should_training_stop = True
        return control

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
free_disk_gib = shutil.disk_usage(OUTPUT_DIR.parent.resolve()).free / 2**30
print(f"Free disk near output: {free_disk_gib:.1f} GiB")
if free_disk_gib < 50:
    print("WARNING: resumable checkpoints may exhaust this disk, especially in full CPT mode.")

last_checkpoint = None
if OUTPUT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
    directory_has_files = any(OUTPUT_DIR.iterdir())
    if directory_has_files and last_checkpoint is None:
        raise RuntimeError(
            f"{OUTPUT_DIR} is non-empty but has no resumable checkpoint. "
            "Choose a new OUTPUT_DIR rather than overwriting it."
        )
    if directory_has_files and not RESUME_IF_AVAILABLE:
        raise RuntimeError(
            f"{OUTPUT_DIR} already contains a run. Set RESUME_IF_AVAILABLE=True or choose a new OUTPUT_DIR."
        )

loader_kwargs = {
    "dataloader_num_workers": DATALOADER_WORKERS,
    "dataloader_pin_memory": True,
    "dataloader_persistent_workers": DATALOADER_WORKERS > 0,
}
if DATALOADER_WORKERS > 0:
    loader_kwargs["dataloader_prefetch_factor"] = 2

checkpointing_kwargs = {}
if USE_GRADIENT_CHECKPOINTING:
    checkpointing_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    overwrite_output_dir=False,
    do_train=True,
    do_eval=RUN_EVALUATION,
    eval_strategy="steps" if RUN_EVALUATION else "no",
    eval_steps=EVAL_STEPS if RUN_EVALUATION else None,
    prediction_loss_only=True,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    optim="adamw_torch_fused",
    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    torch_compile=USE_TORCH_COMPILE,
    torch_compile_mode="default" if USE_TORCH_COMPILE else None,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    save_safetensors=True,
    save_only_model=False,
    report_to="none",
    include_num_input_tokens_seen=True,
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=True,
    use_liger_kernel=LIGER_ACTIVE,
    **loader_kwargs,
    **checkpointing_kwargs,
)

callbacks = []
if TARGET_TRAINING_HOURS is not None:
    callbacks.append(WallClockLimitCallback(TARGET_TRAINING_HOURS))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset if RUN_EVALUATION else None,
    data_collator=default_data_collator,
    processing_class=tokenizer,
    callbacks=callbacks,
)

# Liger patches module classes on the live model when the Trainer is built. That should not
# touch weight references, but the embedding tie is load-bearing for the 25,187 new Sinhala
# tokens, so re-assert it rather than assume.
if TRAINING_MODE == "lora" and NEW_TOKEN_COUNT > 0 and embeddings_are_tied:
    trained_base = trainer.model.base_model.model
    tied_target = trained_base.model.embed_tokens.modules_to_save["default"].weight
    if trained_base.lm_head.weight.data_ptr() != tied_target.data_ptr():
        raise RuntimeError(
            "The lm_head/embedding tie was broken during Trainer construction "
            f"(use_liger_kernel={LIGER_ACTIVE}). New Sinhala tokens would train on the "
            "input side only — do not start training in this state."
        )
    print("Embedding tie re-verified after Trainer construction.")

resume_checkpoint = last_checkpoint if RESUME_IF_AVAILABLE else None
print(f"Attention backend : {getattr(model.config, '_attn_implementation', 'sdpa')}")
print(f"Resume checkpoint : {resume_checkpoint or 'none (new run)'}")
if TARGET_TRAINING_HOURS is not None:
    print(f"Wall-clock backstop: hard stop at {TARGET_TRAINING_HOURS:.1f} h "
          f"(MAX_STEPS={MAX_STEPS:,} should finish first at "
          f"{OBSERVED_STEPS_PER_SECOND} it/s)")
print(f"Estimated updates : {len(train_dataset) // (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS):,} per epoch")

# ---- Per-step memory budget ----
# The logits tensor dominates at this vocabulary size and is the term that OOMs first, so
# print it before committing GPU time rather than discovering it in a traceback.
tokens_per_pass = PER_DEVICE_TRAIN_BATCH_SIZE * SEQUENCE_LENGTH
logits_gib = tokens_per_pass * len(tokenizer) * 4 / 2**30
hidden_size = base_config.hidden_size
layer_count = base_config.num_hidden_layers
per_layer_gib = (
    tokens_per_pass * (3 * base_config.intermediate_size + 6 * hidden_size) * 2 / 2**30
)
if USE_GRADIENT_CHECKPOINTING:
    activation_gib = tokens_per_pass * hidden_size * 2 * layer_count / 2**30 + per_layer_gib
else:
    activation_gib = per_layer_gib * layer_count
visible_gib = torch.cuda.get_device_properties(device_index).total_memory / 2**30

print(f"\nPer-step memory estimate (microbatch {PER_DEVICE_TRAIN_BATCH_SIZE}, {tokens_per_pass:,} tokens/pass):")
if LIGER_ACTIVE:
    print(f"  FP32 logits          : chunked by Liger (would be {2 * logits_gib:.2f} GiB unfused)")
    logits_gib = 0.0
else:
    print(f"  FP32 logits          : {logits_gib:6.2f} GiB (allocated twice: value + gradient)")
print(f"  Activations          : {activation_gib:6.2f} GiB "
      f"({'checkpointed' if USE_GRADIENT_CHECKPOINTING else 'NOT checkpointed'})")
print(f"  Weights/optimizer    : ~{(parameter_bytes + estimated_fp32_adam_bytes + estimated_fp32_gradient_bytes) / 2**30:6.2f} GiB")
print(f"  Visible VRAM         : {visible_gib:6.1f} GiB")
if 2 * logits_gib + activation_gib > 0.6 * visible_gib:
    print("  WARNING: the per-step terms alone exceed 60% of VRAM. Lower "
          "PER_DEVICE_TRAIN_BATCH_SIZE (and raise GRADIENT_ACCUMULATION_STEPS to compensate).")

## 10. Train, evaluate, save, and calculate B200 cost

In [ ]:
import json
import time

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
wall_start = time.perf_counter()

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

wall_runtime_seconds = time.perf_counter() - wall_start
train_metrics = dict(train_result.metrics)
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_state()

eval_metrics = {}
if RUN_EVALUATION:
    eval_metrics = trainer.evaluate()
    trainer.log_metrics("eval", eval_metrics)
    trainer.save_metrics("eval", eval_metrics)

trainer.model.config.use_cache = True
FINAL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(FINAL_ARTIFACT_DIR))
tokenizer.save_pretrained(str(FINAL_ARTIFACT_DIR))

reported_runtime_seconds = float(train_metrics.get("train_runtime", wall_runtime_seconds))
estimated_gpu_cost_usd = reported_runtime_seconds / 3600 * HOURLY_GPU_COST_USD
peak_allocated_gib = torch.cuda.max_memory_allocated() / 2**30
peak_reserved_gib = torch.cuda.max_memory_reserved() / 2**30
input_tokens_seen = int(getattr(trainer.state, "num_input_tokens_seen", 0) or 0)
tokens_per_second = input_tokens_seen / reported_runtime_seconds if input_tokens_seen else None

summary = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "tokenizer_id": TOKENIZER_ID,
    "training_mode": TRAINING_MODE,
    "new_token_count": NEW_TOKEN_COUNT,
    "sequence_length": SEQUENCE_LENGTH,
    "microbatch": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "tokens_per_update": tokens_per_update,
    "trainable_parameters": trainable_parameters,
    "runtime_seconds": reported_runtime_seconds,
    "input_tokens_seen": input_tokens_seen,
    "tokens_per_second": tokens_per_second,
    "peak_allocated_gib": peak_allocated_gib,
    "peak_reserved_gib": peak_reserved_gib,
    "hourly_gpu_cost_usd": HOURLY_GPU_COST_USD,
    "estimated_training_cost_usd": estimated_gpu_cost_usd,
    "train_metrics": train_metrics,
    "eval_metrics": eval_metrics,
}
with (OUTPUT_DIR / "b200_run_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

print(f"Final artifact       : {FINAL_ARTIFACT_DIR.resolve()}")
print(f"Training runtime     : {reported_runtime_seconds / 3600:.3f} h")
print(f"Estimated GPU cost   : ${estimated_gpu_cost_usd:.2f} at ${HOURLY_GPU_COST_USD:.2f}/h")
print(f"Peak allocated VRAM  : {peak_allocated_gib:.1f} GiB")
print(f"Peak reserved VRAM   : {peak_reserved_gib:.1f} GiB")
if tokens_per_second is not None:
    print(f"Training throughput  : {tokens_per_second:,.0f} input tokens/s")

## Interpreting the first run

- **Eval loss should fall steadily** through the epoch; a flat curve usually means the
  learning rate is too low or the new embeddings dominate the gradient budget.
- **Watch the first checkpoints** for throughput and peak memory; if reserved VRAM is far
  below capacity, raising the microbatch is the cheapest speedup.
- **The chat behaviour of the base model will degrade** during raw-text CPT — expected;
  the QA fine-tuning notebook restores task behaviour on top of this checkpoint.

## 11. Quick Sinhala generation check

In [ ]:
generation_model = trainer.model
generation_model.eval()
generation_model.config.use_cache = True

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(generation_model.device)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        output = generation_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    continuation = output[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(continuation, skip_special_tokens=True)

prompts = [
    "ශ්‍රී ලංකාව",
    "බුද්ධ ධර්මය",
    "සිංහල භාෂාව",
]

for prompt in prompts:
    print(f"Prompt : {prompt}")
    print(f"Output : {generate(prompt)}")
    print("-" * 80)

## 12. Push the CPT adapter + tokenizer to Hugging Face

Uploads `FINAL_ARTIFACT_DIR` (LoRA adapter with the trained embedding copy, plus the
extended tokenizer) to `isji/qwen3-4b-cpt`.

**How downstream notebooks must consume this adapter** (tied-embedding caveat): loading
the adapter with `PeftModel.from_pretrained` re-creates the untied state, so always go
through a merge → save → reload cycle before inference or further training — reloading is
what re-ties `lm_head` to the *trained* embeddings (verified):

```python
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("isji/qwen3-4b-cpt")
base.resize_token_embeddings(len(tokenizer), mean_resizing=True)
model = PeftModel.from_pretrained(base, "isji/qwen3-4b-cpt")
model = model.merge_and_unload()
model.save_pretrained("/tmp/qwen3-4b-cpt-merged")          # tied save: one shared matrix
model = AutoModelForCausalLM.from_pretrained("/tmp/qwen3-4b-cpt-merged")  # correctly re-tied
```

The QA fine-tuning notebook (`qwen3-4b-sinhala-qa-finetuning-poc.ipynb`) should point its
`MODEL_ID` at the merged directory (or a pushed merged repo) and load the tokenizer from
this repo.

In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    if not HF_REPO_ID or "/" not in HF_REPO_ID:
        raise ValueError("Set HF_REPO_ID to 'username-or-org/repository' before uploading.")
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=HF_REPO_PRIVATE, exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(FINAL_ARTIFACT_DIR),
        commit_message=(
            f"{TRAINING_MODE} Sinhala CPT of {MODEL_ID} with {TOKENIZER_ID} "
            f"(+{NEW_TOKEN_COUNT:,} tokens) on one B200"
        ),
    )
    print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Upload skipped. Set PUSH_TO_HUB=True in the configuration cell when ready.")